# Graphiti × Red Rising — Prototype

Testing whether Graphiti is viable for a fiction reading companion.
Maps chapter index to temporal axis for spoiler-safe retrieval and trajectory queries.

## Setup

In [ ]:
# pip install graphiti-core[google-genai] ebooklib beautifulsoup4 lxml

import os
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
EPUB_PATH      = "../uploads/red-rising/book_0.epub"
GROUP_ID       = "red_rising_book1"
MAX_CHAPTERS   = 5  # start small, increase once working

os.environ["SEMAPHORE_LIMIT"] = "5"  # lower if hitting Gemini 429s

## Temporal Axis Design

In [2]:
from datetime import datetime, timezone, timedelta

BASE_DATE = datetime(2000, 1, 1, tzinfo=timezone.utc)

def chapter_timestamp(index: int) -> datetime:
    """Map chapter index to fake datetime for Graphiti temporal ordering."""
    return BASE_DATE + timedelta(days=index)

# Example: Ch0 -> 2000-01-01, Ch1 -> 2000-01-02, etc.
for i in range(3):
    print(f"Chapter {i}: {chapter_timestamp(i)}")

Chapter 0: 2000-01-01 00:00:00+00:00
Chapter 1: 2000-01-02 00:00:00+00:00
Chapter 2: 2000-01-03 00:00:00+00:00


## Graphiti Init — Neo4j + Gemini

> Start Neo4j first: `docker compose -f docker-compose.neo4j.yml up -d`

In [5]:
import warnings
import logging
logging.getLogger("neo4j").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

from graphiti_core import Graphiti
from graphiti_core.driver.neo4j_driver import Neo4jDriver
from graphiti_core.llm_client.gemini_client import GeminiClient, LLMConfig
from graphiti_core.embedder.gemini import GeminiEmbedder, GeminiEmbedderConfig
from graphiti_core.cross_encoder.gemini_reranker_client import GeminiRerankerClient
from google.genai import types

# Patch GenerateContentConfig.__init__ to always include BLOCK_NONE safety settings.
# This is needed because graphiti-core doesn't expose safety_settings, and Red Rising
# contains violence that triggers Gemini's default filters.
_orig_config_init = types.GenerateContentConfig.__init__

def _patched_config_init(self, **kwargs):
    kwargs.setdefault("safety_settings", [
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT",        threshold="BLOCK_NONE"),
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH",       threshold="BLOCK_NONE"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
    ])
    _orig_config_init(self, **kwargs)

types.GenerateContentConfig.__init__ = _patched_config_init

# Neo4j: start with `docker compose -f docker-compose.neo4j.yml up -d`
graphiti = Graphiti(
    graph_driver=Neo4jDriver(
        uri="bolt://localhost:7687",
        user="neo4j",
        password="",
    ),
    llm_client=GeminiClient(config=LLMConfig(api_key=GOOGLE_API_KEY, model="gemini-2.5-flash-lite")),
    embedder=GeminiEmbedder(config=GeminiEmbedderConfig(api_key=GOOGLE_API_KEY, embedding_model="gemini-embedding-001")),
    cross_encoder=GeminiRerankerClient(config=LLMConfig(api_key=GOOGLE_API_KEY, model="gemini-2.5-flash-lite")),
)
await graphiti.build_indices_and_constraints()
print("Graphiti initialized with Neo4j.")

Graphiti initialized with Neo4j.


## Ingestion

In [7]:
import sys
sys.path.append("../src")

In [9]:
from graphiti_core.nodes import EpisodeType
from src.ingestion.epub_parser import parse_epub

CHUNK_WORDS = 800  # keeps LLM JSON output small enough to avoid truncation

def chunk_text(text: str, max_words: int) -> list[str]:
    """Split text into chunks of max_words with no overlap."""
    words = text.split()
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]

chapters = parse_epub(EPUB_PATH)[:MAX_CHAPTERS]

for chapter in chapters:
    chunks = chunk_text(chapter.text, CHUNK_WORDS)
    n = len(chunks)
    print(f"[{chapter.index}] {chapter.label} ({len(chapter.text.split())} words, {n} chunk(s))...", end=" ")
    for i, chunk in enumerate(chunks):
        await graphiti.add_episode(
            name=f"Ch{chapter.index}.{i}: {chapter.label}",
            episode_body=chunk,
            source=EpisodeType.text,
            reference_time=chapter_timestamp(chapter.index),
            source_description=f"Red Rising Book 1 — {chapter.label} (part {i+1}/{n})",
            group_id=GROUP_ID,
        )
    print("done")

print(f"\nIngested {len(chapters)} chapters.")

[0] Prologue (269 words, 1 chunk(s))... done
[1] 1: Helldiver (2260 words, 3 chunk(s))... 

Error in generating LLM response: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying after application error (attempt 1/2): 
Target entity not found in nodes for edge relation: IS_MARRIED_TO
Target entity not found in nodes for edge relation: IS_CHILDHOOD_SWEETHEART_OF
Target entity not found in nodes for edge relation: SUPERVISES
Target entity not found in nodes for edge relation: MOCKS
Target entity not found in nodes for edge relation: ADVISES
Target entity not found in nodes for edge relation: IS_RELATED_TO
Target entity not found in nodes for edge relation: IS_HEAD_TALK_OF
Target entity not found in nodes for edge relation: ACCUSES
Target entity not found in nodes for edge relation: IS_CAUTIOUS_AND_IMMODERATE_IN_HIS
Target entity not found in nodes for edge relation: IS_OF_CLAN
Target entity not found in nodes for edge relation: H

done
[2] 2: The Township (3417 words, 5 chunk(s))... 

Error in generating LLM response: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying after application error (attempt 1/2): 
Error in generating LLM response: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying after application error (attempt 1/2): 
Error in generating LLM response: Server disconnected without sending a response.
Retrying after application error (attempt 1/2): 
Target entity not found in nodes for edge relation: USES
Target entity not found in nodes for edge relation: GIVES_SUGAR_AND_FRUIT_TO
Target entity not found in nodes for edge relation: GETS
Target entity not found in nodes for edge relation: GETS
Source entity not found in nodes for edge relation: HAS
Target en

done
[3] 3: The Laurel (2801 words, 4 chunk(s))... 

Target entity not found in nodes for edge relation: IS_HURT_BY
Target entity not found in nodes for edge relation: TAKES_PLACE_IN
Target entity not found in nodes for edge relation: PLAYS_FOR
Target entity not found in nodes for edge relation: PLAYS
Target entity not found in nodes for edge relation: TAUGHT
Target entity not found in nodes for edge relation: IS_TEACHER_OF
Target entity not found in nodes for edge relation: TAUGHT_DANCES_TO
Source entity not found in nodes for edge relation: BESTED_OTHERS_IN_GAMES_WITH_LESSONS_FROM
Target entity not found in nodes for edge relation: DRILL_ALONE
Source entity not found in nodes for edge relation: COMPARED_TO
Target entity not found in nodes for edge relation: IS_FROM
Target entity not found in nodes for edge relation: TUCKS_INTO_SKIRTS
Source entity not found in nodes for edge relation: IS_GRINNING_LIKE_A_FOOL_AT
Target entity not found in nodes for edge relation: IS_OUTPACED_BY
Source entity not found in nodes for edge relation: IS_OUTP

RateLimitError: Rate limit exceeded. Please try again later.

## Search Configs

In [3]:
from graphiti_core.search.search_filters import SearchFilters, DateFilter, ComparisonOperator

def at_chapter(boundary: int) -> SearchFilters:
    """Return a SearchFilters that scopes results to edges valid at or before chapter `boundary`."""
    return SearchFilters(
        valid_at=[[DateFilter(
            date=chapter_timestamp(boundary),
            comparison_operator=ComparisonOperator.less_than_equal,
        )]]
    )

## Test A: Spoiler-safe Retrieval

In [6]:
async def query_at_chapter(query, chapter_boundary, n=10):
    return await graphiti.search(
        query=query,
        group_ids=[GROUP_ID],
        num_results=n,
        search_filter=at_chapter(chapter_boundary),
    )

early = await query_at_chapter("what is happening with Eo", chapter_boundary=1)
late  = await query_at_chapter("what is happening with Eo", chapter_boundary=4)

print("\n=== TEST A: Spoiler-safe Retrieval ===")
print(f"\nEarly query (Chapter ≤3, {len(early)} results):")
for e in early[:5]:
    print(f"  {getattr(e, 'source_node_name', '?')} -> {getattr(e, 'target_node_name', '?')}: {getattr(e, 'fact', '?')}")

print(f"\nLate query (Chapter ≤9, {len(late)} results):")
for e in late[:5]:
    print(f"  {getattr(e, 'source_node_name', '?')} -> {getattr(e, 'target_node_name', '?')}: {getattr(e, 'fact', '?')}")

early_facts = {getattr(e, 'fact', '') for e in early}
late_facts  = {getattr(e, 'fact', '') for e in late}
spoiler_facts = late_facts - early_facts

if spoiler_facts:
    print(f"\n✓ Spoiler gate working: {len(spoiler_facts)} facts hidden in early query")
    test_a_result = "Y"
else:
    print(f"\n✗ Spoiler gate not detecting differences")
    test_a_result = "N"


=== TEST A: Spoiler-safe Retrieval ===

Early query (Chapter ≤3, 0 results):

Late query (Chapter ≤9, 0 results):

✗ Spoiler gate not detecting differences


In [16]:
early

[]

## Test B: Trajectory Query

In [11]:
edges = await graphiti.search(
    query="Darrow journey events decisions actions",
    group_ids=[GROUP_ID],
    num_results=25,
    search_filter=at_chapter(MAX_CHAPTERS),
)
edges.sort(key=lambda e: getattr(e, "valid_at", BASE_DATE) or BASE_DATE)

print("\n=== TEST B: Trajectory Query ===")
print(f"\nDarrow timeline ({len(edges)} edges):")
for e in edges[:15]:
    valid_at = getattr(e, "valid_at", BASE_DATE) or BASE_DATE
    print(f"  {valid_at.date()}: {getattr(e, 'fact', '?')}")

unique_chapters = len(set(
    (getattr(e, "valid_at", BASE_DATE) or BASE_DATE).day
    for e in edges if getattr(e, "valid_at", None)
))

if unique_chapters >= 3 and len(edges) > 5:
    print(f"\n✓ Trajectory is coherent: {len(edges)} edges across {unique_chapters} chapters")
    test_b_result = "Y"
else:
    print(f"\n✗ Trajectory unclear: only {len(edges)} edges, {unique_chapters} chapters")
    test_b_result = "N"


=== TEST B: Trajectory Query ===

Darrow timeline (0 edges):

✗ Trajectory unclear: only 0 edges, 0 chapters


## Test C: Belief/State Revision

In [12]:
edges = await graphiti.search(
    query="loyalty betrayal alliance identity death",
    group_ids=[GROUP_ID],
    num_results=30,
    search_filter=at_chapter(MAX_CHAPTERS),
)
invalidated = [e for e in edges if getattr(e, "invalid_at", None) is not None]

print("\n=== TEST C: Belief/State Revision ===")
print(f"\nTotal edges: {len(edges)}")
print(f"Invalidated edges: {len(invalidated)}")

if invalidated:
    print(f"\nInvalidated facts (state-change events):")
    for e in invalidated[:10]:
        fact = getattr(e, "fact", "?")
        valid_at = getattr(e, "valid_at", None)
        invalid_at = getattr(e, "invalid_at", None)
        print(f"  {fact}")
        print(f"    valid: {valid_at}, invalid: {invalid_at}")
    print(f"\n✓ Belief revision detected: {len(invalidated)} contradictions")
    test_c_result = "Y"
else:
    print(f"\n⚠ No invalidated edges found. Try MAX_CHAPTERS=20+ if unexpected.")
    test_c_result = "N"


=== TEST C: Belief/State Revision ===

Total edges: 0
Invalidated edges: 0

⚠ No invalidated edges found. Try MAX_CHAPTERS=20+ if unexpected.


## Test D: Entity Dedup

In [13]:
from graphiti_core.search.search_config_recipes import NODE_HYBRID_SEARCH_RRF

results = await graphiti.search_(
    query="Darrow Helldiver Lykos Red miner protagonist",
    config=NODE_HYBRID_SEARCH_RRF,
    group_ids=[GROUP_ID],
)
nodes = results.nodes

unique_uuids = set(getattr(n, "uuid", None) for n in nodes if getattr(n, "uuid", None))

print("\n=== TEST D: Entity Dedup ===")
print(f"\nNodes found: {len(nodes)}")
print(f"Unique UUIDs: {len(unique_uuids)}")

print(f"\nEntities:")
for n in nodes[:10]:
    name = getattr(n, "name", "?")
    summary = getattr(n, "summary", "?")
    print(f"  [{getattr(n, 'uuid', '?')[:8]}] {name}: {str(summary)[:80] if summary else '(no summary)'}")

if len(unique_uuids) == 1:
    print(f"\n✓ Entity dedup perfect: all Darrow aliases → 1 node")
    test_d_result = "Y"
elif len(unique_uuids) <= 2:
    print(f"\n⚠ Entity dedup partial: {len(unique_uuids)} nodes (expected 1)")
    test_d_result = "Y"
else:
    print(f"\n✗ Entity dedup failed: {len(unique_uuids)} nodes (expected 1)")
    test_d_result = "N"


=== TEST D: Entity Dedup ===

Nodes found: 0
Unique UUIDs: 0

Entities:

⚠ Entity dedup partial: 0 nodes (expected 1)


## Score Sheet

In [14]:
print("\n" + "="*60)
print("GRAPHITI PROTOTYPE SCORECARD")
print("="*60)
print(f"""
A. Spoiler-safe retrieval
   Early query excluded later facts?              {test_a_result}

B. Trajectory query
   Edge timeline tells coherent arc?             {test_b_result}

C. Belief/state revision
   Invalidated edges found?                      {test_c_result}

D. Entity dedup
   Darrow aliases → 1 node?                      {test_d_result}

---

VERDICT: Worth building Phase 1+2 custom
         extraction on top?                      ?

""")
print("="*60)


GRAPHITI PROTOTYPE SCORECARD

A. Spoiler-safe retrieval
   Early query excluded later facts?              N

B. Trajectory query
   Edge timeline tells coherent arc?             N

C. Belief/state revision
   Invalidated edges found?                      N

D. Entity dedup
   Darrow aliases → 1 node?                      Y

---

VERDICT: Worth building Phase 1+2 custom
         extraction on top?                      ?




# Iteration 2

In [10]:
import asyncio
import os
import json
from datetime import datetime, timedelta

# --- NEW SDK IMPORT ---
# pip install google-genai
from google import genai
from google.genai import types

# --- YOUR LOCAL MODULES ---
from src.ingestion.epub_parser import parse_epub

# --- GRAPHITI & DB ---
from graphiti_core import Graphiti
from graphiti_core.nodes import EpisodeType

# --- CONFIG ---
API_KEY = os.getenv("GOOGLE_API_KEY")
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "password"
EPUB_PATH      = "../uploads/red-rising/book_0.epub"

# The new standard for high-volume extraction
MODEL_ID = "gemini-3.1-flash-lite"

# Initialize the new Client
client = genai.Client(api_key=API_KEY)

async def extract_chapter_json(chapter_text: str, label: str):
    """
    Uses the modern Client.models.generate_content pattern.
    Optimized for high-throughput extraction.
    """
    prompt = f"""
    Analyze the text from chapter: {label}
    Extract key story entities (Characters, Factions, Locations) and their Relationships.
    
    RULES:
    1. Focus on durable state changes (e.g., 'Darrow joins the Sons of Ares').
    2. Ensure all 'target_entity_name' values have a matching entry in 'entities'.
    3. Output strictly valid JSON.
    
    TEXT:
    {chapter_text[:25000]} # Gemini 3.1 easily handles larger windows
    """

    try:
        # Modern SDK uses config objects for controlled output
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            )
        )
        return json.loads(response.text)
    except Exception as e:
        print(f"Extraction error for {label}: {e}")
        return None

async def main():
    # 1. Initialize Graphiti
    graphiti = Graphiti(NEO4J_URI, NEO4J_USER, NEO4J_PASS)
    await graphiti.build_indices_and_constraints()

    # 2. Use your custom epub_parser
    print("Parsing EPUB...")
    chapters = parse_epub(EPUB_PATH)
    print(f"Parsed {len(chapters)} chapters.")

    # 3. Sequential processing to respect Priority Tier throughput
    for i, chapter in enumerate(chapters):
        print(f"Processing: {chapter.label}")
        
        chapter_data = await extract_chapter_json(chapter.text, chapter.label)
        
        if chapter_data:
            # Map chapter order to temporal dates for the Spoiler Shield
            ref_time = datetime(2026, 1, 1) + timedelta(days=i)
            
            # Ingesting the JSON directly into Graphiti
            await graphiti.add_episode(
                name=chapter.label,
                episode_body=chapter_data, 
                source=EpisodeType.json,
                reference_time=ref_time,
                source_description=f"Automated extraction from chapter: {chapter.label}"
            )
            print(f"Done: {chapter.label}")

        # Minor throttle for stability
        await asyncio.sleep(1.5)

    await graphiti.close()
    print("Graph Build Complete.")

if __name__ == "__main__":
    asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop